# Máquina de Soporte Vectorial para regresión (SVR)

Implementación basada en la sección **SVM para Regresión** del ejemplo de clase `Modelos de ejemplo/2026_10_22_SVM.ipynb`. Como `log_bought` es continuo, usamos SVR y no SVC de clasificación. Se comparan kernel lineal y RBF y se prueban distintos valores de `C`, `epsilon` y `gamma`.

La selección de hiperparámetros usa MAE en escala logarítmica y validación cruzada por grupos. Se reutiliza el test común de KNN para mantener la comparación entre modelos. El escalado y la imputación se ajustan dentro de cada fold.

## 1. Librerías y rutas

Ejecuta este notebook desde la raíz del proyecto o desde `Modelos entrenados/`. Primero debe existir `Models_Results/knn/split_indices.json`, generado por el experimento KNN.

In [1]:
from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
TARGET = "log_bought"
VARIANTS = (1, 2, 3, 4)
SEARCH_SAMPLE_SIZE = 2000
CV_FOLDS = 4

def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / "Limpieza de datos").is_dir() and (folder / "Modelos entrenados").is_dir():
            return folder
    raise FileNotFoundError("No se encontró la raíz del proyecto Amazon product sales.")

ROOT = find_project_root()
MODEL_DIR = ROOT / "Modelos entrenados"
RESULTS_DIR = MODEL_DIR / "Models_Results" / "svm"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = MODEL_DIR / "Models_Results" / "knn" / "split_indices.json"
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"No existe {SPLIT_PATH}. Ejecuta primero el notebook KNN.")

print(f"Raíz del proyecto: {ROOT}")
print(f"Resultados: {RESULTS_DIR}")

Raíz del proyecto: /home/barrera/Documentos/Udea/2026-2/Modelos y simulación/Trabajo de amazon/Amazon product sales
Resultados: /home/barrera/Documentos/Udea/2026-2/Modelos y simulación/Trabajo de amazon/Amazon product sales/Modelos entrenados/Models_Results/svm


## 2. Carga de datos y partición común

Se conserva el test reservado por KNN. La búsqueda y el ajuste final usan únicamente las filas restantes. La búsqueda de hiperparámetros se hace sobre una muestra de grupos de train para controlar el costo computacional de SVR; los modelos finales sí se ajustan con todas las filas de train.

In [2]:
with SPLIT_PATH.open(encoding="utf-8") as file:
    saved_splits = json.load(file)

datasets = {}
for variant in VARIANTS:
    label = f"v{variant}"
    path = ROOT / "Limpieza de datos" / f"dataset_{label}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No existe el dataset requerido: {path}")
    frame = pd.read_csv(path)
    if label not in saved_splits or "test" not in saved_splits[label]:
        raise KeyError(f"No se encontró el índice de test para {label}.")
    test_idx = np.asarray(saved_splits[label]["test"], dtype=int)
    if test_idx.size == 0 or test_idx.min() < 0 or test_idx.max() >= len(frame):
        raise ValueError(f"Índices de test no válidos para {label}.")
    train_idx = np.setdiff1d(np.arange(len(frame)), test_idx)
    datasets[variant] = {"frame": frame, "train_idx": train_idx, "test_idx": test_idx}
    print(f"{label}: filas={len(frame):,} | train={len(train_idx):,} | test={len(test_idx):,}")

if len({len(data["test_idx"]) for data in datasets.values()}) != 1:
    raise ValueError("El número de filas de test no coincide entre las variantes.")

v1: filas=24,436 | train=19,549 | test=4,887


v2: filas=24,436 | train=19,549 | test=4,887


v3: filas=31,202 | train=26,315 | test=4,887


v4: filas=31,202 | train=26,315 | test=4,887


## 3. Características sin fuga de información

Se excluyen del predictor el objetivo `log_bought`, `bought_in_last_month` y la categoría auxiliar `bought_cat`. Cuando existen variables originales y transformadas logarítmicamente, se usa solo la versión log. `is_price_outlier` se omite para seguir la selección de características del KNN. `buy_box_status` se convierte a indicador de valor desconocido. Las variantes v1/v3 incluyen además `log_delivery_cost`.

In [3]:
BASE_FEATURES = [
    "Rating", "has_discount", "is_best_seller_flag", "is_sponsored_flag",
    "has_coupon", "is_free_delivery", "has_sustainability_badge",
    "log_price", "log_reviews", "log_title_words", "buy_box_unknown",
]
LOG_PAIRS = {
    "current_price": "log_price",
    "Number_of_reviews": "log_reviews",
    "title_word_count": "log_title_words",
    "delivery_cost": "log_delivery_cost",
}

def build_xy(frame):
    required = {TARGET, "buy_box_status"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Faltan columnas requeridas: {required - set(frame.columns)}")
    data = frame.copy()
    data["buy_box_unknown"] = (data["buy_box_status"] == "unknown").astype(int)
    features = [column for column in BASE_FEATURES if column in data.columns]
    if "log_delivery_cost" in data.columns:
        features.append("log_delivery_cost")
    excluded = {TARGET, "bought_in_last_month", "bought_cat", "is_price_outlier", "buy_box_status"}
    excluded.update(raw for raw, logged in LOG_PAIRS.items()
                    if raw in data.columns and logged in data.columns)
    if set(features) & excluded:
        raise ValueError(f"Features excluidas/fuga: {set(features) & excluded}")
    if not features:
        raise ValueError("No se encontraron características para entrenar SVR.")
    X = data[features].replace([np.inf, -np.inf], np.nan)
    y = data[TARGET].to_numpy(dtype=float)
    if not np.isfinite(y).all():
        raise ValueError("El objetivo contiene valores no finitos.")
    return X, y, features

for variant, item in datasets.items():
    item["X"], item["y"], item["features"] = build_xy(item["frame"])

group_features = [
    feature for feature in datasets[1]["features"]
    if all(feature in datasets[variant]["features"] for variant in VARIANTS)
]
if not group_features:
    raise ValueError("No hay features comunes para identificar grupos de productos.")

def make_groups(X):
    return pd.util.hash_pandas_object(X[group_features], index=False).to_numpy()

for variant, item in datasets.items():
    item["groups"] = make_groups(item["X"])
    train_groups = np.unique(item["groups"][item["train_idx"]])
    test_groups = np.unique(item["groups"][item["test_idx"]])
    if np.intersect1d(train_groups, test_groups).size:
        raise ValueError(f"Hay grupos repetidos entre train y test en v{variant}.")
    print(f"v{variant}: {len(item['features'])} features; grupos train/test disjuntos.")

test_signatures = []
for item in datasets.values():
    idx = item["test_idx"]
    signature_data = pd.DataFrame({
        "group": item["groups"][idx], "target": item["y"][idx]
    })
    test_signatures.append(
        pd.util.hash_pandas_object(signature_data, index=False).value_counts().sort_index()
    )
if any(not test_signatures[0].equals(signature) for signature in test_signatures[1:]):
    raise ValueError("Los índices guardados no representan el mismo test en las cuatro variantes.")
print("Verificado: las cuatro variantes contienen los mismos productos y objetivos en test.")

display(pd.DataFrame({f"v{v}": pd.Series(datasets[v]["features"]) for v in VARIANTS}))

v1: 12 features; grupos train/test disjuntos.
v2: 11 features; grupos train/test disjuntos.
v3: 12 features; grupos train/test disjuntos.
v4: 11 features; grupos train/test disjuntos.
Verificado: las cuatro variantes contienen los mismos productos y objetivos en test.


                          v1                        v2  \
0                     Rating                    Rating   
1               has_discount              has_discount   
2        is_best_seller_flag       is_best_seller_flag   
3          is_sponsored_flag         is_sponsored_flag   
4                 has_coupon                has_coupon   
5           is_free_delivery          is_free_delivery   
6   has_sustainability_badge  has_sustainability_badge   
7                  log_price                 log_price   
8                log_reviews               log_reviews   
9            log_title_words           log_title_words   
10           buy_box_unknown           buy_box_unknown   
11         log_delivery_cost                       NaN   

                          v3                        v4  
0                     Rating                    Rating  
1               has_discount              has_discount  
2        is_best_seller_flag       is_best_seller_flag  
3          is_spo

## 4. Búsqueda de hiperparámetros

Esta etapa refina `C`, `epsilon` y `gamma` alrededor de la región ganadora anterior y usa el kernel RBF, ganador en las cuatro variantes. Se prueban 45 combinaciones por variante con 4 folds agrupados. La selección usa solo validación cruzada; las características, imputación, escalado y partición se mantienen sin cambios.

In [4]:
PARAM_GRID = {
    "svr__kernel": ["rbf"],
    "svr__C": [0.01, 0.025, 0.05, 0.075, 0.1],
    "svr__epsilon": [0.025, 0.05, 0.075],
    "svr__gamma": [0.005, 0.01, 0.015],
}

def sample_groups(train_idx, groups, size, seed):
    rng = np.random.default_rng(seed)
    rows_by_group = {}
    for row, group in zip(train_idx, groups[train_idx]):
        rows_by_group.setdefault(group, []).append(row)
    chosen, seen, count = [], set(), 0
    for row in rng.permutation(train_idx):
        group = groups[row]
        if group in seen:
            continue
        seen.add(group)
        rows = np.asarray(rows_by_group[group], dtype=int)
        chosen.append(rows)
        count += len(rows)
        if count >= size:
            break
    if not chosen:
        raise ValueError("No se encontraron grupos para la búsqueda.")
    return np.sort(np.concatenate(chosen))

search_rows, best_params, validation_metrics, search_seconds = [], {}, {}, {}
for variant, item in datasets.items():
    label = f"v{variant}"
    sample_idx = sample_groups(item["train_idx"], item["groups"], SEARCH_SAMPLE_SIZE, SEED + variant)
    sample_groups_array = item["groups"][sample_idx]
    strata = item["frame"].loc[sample_idx, "bought_cat"].to_numpy()
    cv = list(StratifiedGroupKFold(
        n_splits=CV_FOLDS, shuffle=True, random_state=SEED
    ).split(item["X"].iloc[sample_idx], strata, sample_groups_array))
    for train_fold, valid_fold in cv:
        if np.intersect1d(sample_groups_array[train_fold], sample_groups_array[valid_fold]).size:
            raise ValueError(f"Fuga por grupos detectada en CV de {label}.")

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("svr", SVR(cache_size=512)),
    ])
    search = GridSearchCV(
        pipeline, PARAM_GRID, scoring="neg_mean_absolute_error", cv=cv,
        n_jobs=1, refit=False, return_train_score=False,
    )
    started = time.perf_counter()
    search.fit(item["X"].iloc[sample_idx], item["y"][sample_idx])
    search_seconds[label] = time.perf_counter() - started
    results = pd.DataFrame(search.cv_results_)
    results["variante"] = label
    results["MAE_CV_log"] = -results["mean_test_score"]
    results["std_CV_log"] = results["std_test_score"]
    search_rows.append(results)
    best_params[label] = search.best_params_
    best_pos = search.best_index_
    validation_metrics[label] = {
        "MAE_log_mean": float(-search.best_score_),
        "MAE_log_std": float(search.cv_results_["std_test_score"][best_pos]),
        "sample_rows": int(len(sample_idx)), "folds": CV_FOLDS,
    }
    print(
        f"{label}: MAE CV log={validation_metrics[label]['MAE_log_mean']:.4f} "
        f"(+/- {validation_metrics[label]['MAE_log_std']:.4f}); "
        f"params={best_params[label]}; n={len(sample_idx):,}; "
        f"tiempo={search_seconds[label]:.1f}s"
    )

search_results = pd.concat(search_rows, ignore_index=True)
search_results.to_csv(RESULTS_DIR / "hyperparameter_search.csv", index=False)
validation_summary = pd.DataFrame([
    {"variante": label, **validation_metrics[label],
     **{key.removeprefix("svr__"): value for key, value in best_params[label].items()}}
    for label in best_params
])
validation_summary.to_csv(RESULTS_DIR / "validation_summary.csv", index=False)
display(validation_summary)
display(search_results[["variante", "params", "MAE_CV_log", "std_CV_log"]]
        .sort_values(["variante", "MAE_CV_log"]))

v1: MAE CV log=0.2865 (+/- 0.0608); params={'svr__C': 0.05, 'svr__epsilon': 0.025, 'svr__gamma': 0.015, 'svr__kernel': 'rbf'}; n=2,089; tiempo=29.2s


v2: MAE CV log=0.2255 (+/- 0.0219); params={'svr__C': 0.075, 'svr__epsilon': 0.025, 'svr__gamma': 0.015, 'svr__kernel': 'rbf'}; n=2,066; tiempo=29.4s


v3: MAE CV log=0.4896 (+/- 0.1834); params={'svr__C': 0.01, 'svr__epsilon': 0.025, 'svr__gamma': 0.015, 'svr__kernel': 'rbf'}; n=2,105; tiempo=27.6s


v4: MAE CV log=0.3760 (+/- 0.1642); params={'svr__C': 0.05, 'svr__epsilon': 0.05, 'svr__gamma': 0.005, 'svr__kernel': 'rbf'}; n=2,011; tiempo=26.3s


  variante  MAE_log_mean  MAE_log_std  sample_rows  folds      C  epsilon  \
0       v1      0.286466     0.060816         2089      4  0.050    0.025   
1       v2      0.225483     0.021900         2066      4  0.075    0.025   
2       v3      0.489578     0.183410         2105      4  0.010    0.025   
3       v4      0.375986     0.164238         2011      4  0.050    0.050   

   gamma kernel  
0  0.015    rbf  
1  0.015    rbf  
2  0.015    rbf  
3  0.005    rbf  

    variante                                             params  MAE_CV_log  \
20        v1  {'svr__C': 0.05, 'svr__epsilon': 0.025, 'svr__...    0.286466   
22        v1  {'svr__C': 0.05, 'svr__epsilon': 0.05, 'svr__g...    0.295201   
19        v1  {'svr__C': 0.05, 'svr__epsilon': 0.025, 'svr__...    0.296955   
28        v1  {'svr__C': 0.075, 'svr__epsilon': 0.025, 'svr_...    0.297925   
29        v1  {'svr__C': 0.075, 'svr__epsilon': 0.025, 'svr_...    0.300364   
..       ...                                                ...         ...   
167       v4  {'svr__C': 0.075, 'svr__epsilon': 0.05, 'svr__...    0.510834   
170       v4  {'svr__C': 0.075, 'svr__epsilon': 0.075, 'svr_...    0.515882   
179       v4  {'svr__C': 0.1, 'svr__epsilon': 0.075, 'svr__g...    0.545618   
176       v4  {'svr__C': 0.1, 'svr__epsilon': 0.05, 'svr__ga...    0.550021   
173       v4  {'svr__C': 0.1, 'svr__epsilon': 0.025, 'svr__g...    0.554301   

     std_CV_log  
20     0.060816  
22     0.048414

## 5. Ajuste final y métricas de test

El test se evalúa una sola vez después de seleccionar los parámetros. Se muestran MAE, RMSE y R² en escala logarítmica, y MAE, RMSE, R² y RMSLE después de aplicar `expm1`. Las predicciones en unidades se limitan a cero para evitar cantidades negativas.

In [5]:
def evaluate(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.maximum(0, np.expm1(y_pred_log))
    return {
        "MAE_log": float(mean_absolute_error(y_true_log, y_pred_log)),
        "RMSE_log": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "R2_log": float(r2_score(y_true_log, y_pred_log)),
        "MAE_unidades": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_unidades": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2_unidades": float(r2_score(y_true, y_pred)),
        "RMSLE_unidades": float(np.sqrt(mean_squared_log_error(y_true, y_pred))),
    }

test_metrics, support_vectors, fit_seconds = {}, {}, {}
prediction_frames, final_models = [], {}
for variant, item in datasets.items():
    label = f"v{variant}"
    params = {key.removeprefix("svr__"): value for key, value in best_params[label].items()}
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("svr", SVR(cache_size=512, **params)),
    ])
    started = time.perf_counter()
    model.fit(item["X"].iloc[item["train_idx"]], item["y"][item["train_idx"]])
    fit_seconds[label] = time.perf_counter() - started
    pred_log = model.predict(item["X"].iloc[item["test_idx"]])
    true_log = item["y"][item["test_idx"]]
    test_metrics[label] = evaluate(true_log, pred_log)
    support_vectors[label] = int(len(model.named_steps["svr"].support_))
    final_models[label] = model
    joblib.dump(model, RESULTS_DIR / f"svr_{label}_final.joblib")

    pred = item["frame"].iloc[item["test_idx"]][["bought_cat"]].copy()
    pred.insert(0, "dataset_row", item["test_idx"])
    pred["variante"] = label
    pred["y_true_log"] = true_log
    pred["y_pred_log"] = pred_log
    pred["y_true_units"] = np.expm1(true_log)
    pred["y_pred_units"] = np.maximum(0, np.expm1(pred_log))
    pred["residual_log"] = true_log - pred_log
    prediction_frames.append(pred)
    print(
        f"{label}: MAE_log={test_metrics[label]['MAE_log']:.4f} | "
        f"RMSE_log={test_metrics[label]['RMSE_log']:.4f} | "
        f"R2_log={test_metrics[label]['R2_log']:.4f} | "
        f"MAE_unidades={test_metrics[label]['MAE_unidades']:.1f} | "
        f"SV={support_vectors[label]:,} | ajuste={fit_seconds[label]:.1f}s"
    )

test_predictions = pd.concat(prediction_frames, ignore_index=True)
test_predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
test_summary = pd.DataFrame([
    {"variante": label, **test_metrics[label], "n_support_vectors": support_vectors[label]}
    for label in test_metrics
]).set_index("variante")
display(test_summary)

v1: MAE_log=0.4834 | RMSE_log=0.7482 | R2_log=0.4720 | MAE_unidades=527.8 | SV=16,266 | ajuste=27.2s


v2: MAE_log=0.4836 | RMSE_log=0.7446 | R2_log=0.4770 | MAE_unidades=507.7 | SV=16,287 | ajuste=26.1s


v3: MAE_log=0.4844 | RMSE_log=0.7706 | R2_log=0.4398 | MAE_unidades=545.5 | SV=23,239 | ajuste=52.1s


v4: MAE_log=0.5041 | RMSE_log=0.7769 | R2_log=0.4306 | MAE_unidades=530.4 | SV=22,430 | ajuste=50.0s


           MAE_log  RMSE_log    R2_log  MAE_unidades  RMSE_unidades  \
variante                                                              
v1        0.483397  0.748197  0.471965    527.788277    3945.905307   
v2        0.483557  0.744627  0.476991    507.748857    3867.043652   
v3        0.484408  0.770621  0.439838    545.525245    4090.548674   
v4        0.504074  0.776948  0.430603    530.424544    3951.561257   

          R2_unidades  RMSLE_unidades  n_support_vectors  
variante                                                  
v1           0.085638        0.748197              16266  
v2           0.121821        0.744627              16287  
v3           0.017374        0.770621              23239  
v4           0.083014        0.776948              22430  

## 6. Gráficas y análisis de error

Se generan gráficas del mejor MAE de validación por kernel, valores reales frente a predichos y errores por categoría de ventas. Los archivos tabulares y figuras se guardan en `Modelos entrenados/Models_Results/svm/` para documentar y reutilizar las salidas.

In [6]:
kernel_scores = search_results.groupby(
    ["variante", "param_svr__kernel"], as_index=False
)["MAE_CV_log"].min()
fig, ax = plt.subplots(figsize=(9, 5))
labels = [f"v{v}" for v in VARIANTS]
positions = np.arange(len(labels))
width = 0.35
for offset, kernel in enumerate(("linear", "rbf")):
    values = [
        float(kernel_scores.loc[
            (kernel_scores["variante"] == label)
            & (kernel_scores["param_svr__kernel"] == kernel), "MAE_CV_log"
        ].min())
        for label in labels
    ]
    ax.bar(positions + (offset - 0.5) * width, values, width, label=kernel)
ax.set_xticks(positions, labels)
ax.set_ylabel("MAE de validación cruzada (log)")
ax.set_title("Mejor MAE de CV por kernel y variante")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Kernel")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cv_mae_by_kernel.png", dpi=160)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, label in zip(axes.flat, labels):
    subset = test_predictions[test_predictions["variante"] == label]
    ax.scatter(subset["y_true_log"], subset["y_pred_log"], alpha=0.25, s=12)
    low = min(subset["y_true_log"].min(), subset["y_pred_log"].min())
    high = max(subset["y_true_log"].max(), subset["y_pred_log"].max())
    ax.plot([low, high], [low, high], "k--", linewidth=1)
    ax.set(title=label, xlabel="Real: log_bought", ylabel="Predicho: log_bought")
    ax.grid(alpha=0.2)
fig.suptitle("SVR: valores reales vs. predichos en test")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_test_real_vs_pred.png", dpi=160)
plt.show()

error_by_category = test_predictions.groupby(
    ["variante", "bought_cat"], observed=True
).apply(lambda part: pd.Series({
    "n": len(part),
    "MAE_log": mean_absolute_error(part["y_true_log"], part["y_pred_log"]),
    "MAE_unidades": mean_absolute_error(part["y_true_units"], part["y_pred_units"]),
}), include_groups=False).reset_index()
error_by_category.to_csv(RESULTS_DIR / "error_by_sales_category.csv", index=False)
display(error_by_category)

<Figure size 900x500 with 1 Axes>

<Figure size 1200x1000 with 4 Axes>

   variante bought_cat       n   MAE_log  MAE_unidades
0        v1    101-200  1326.0  0.325894     62.445814
1        v1    201-500  1655.0  0.342353    111.459228
2        v1     50-100   765.0  0.352387     45.796177
3        v1       >500  1141.0  0.958858   1995.617058
4        v2    101-200  1326.0  0.345298     65.367806
5        v2    201-500  1655.0  0.345572    113.447588
6        v2     50-100   765.0  0.343489     46.596441
7        v2       >500  1141.0  0.938287   1902.970129
8        v3    101-200  1326.0  0.337053     63.804771
9        v3    201-500  1655.0  0.311651     92.303044
10       v3     50-100   765.0  0.377653     44.274363
11       v3       >500  1141.0  0.977809   2098.812726
12       v4    101-200  1326.0  0.388704     82.367607
13       v4    201-500  1655.0  0.320073     97.946155
14       v4     50-100   765.0  0.383919     59.881662
15       v4       >500  1141.0  0.985600   1993.913182

## 7. Exportación, limitaciones e interpretación

El resumen de métricas y decisiones queda en `metrics.json`; la búsqueda completa, parámetros seleccionados, predicciones, errores por categoría y modelos serializados también quedan en `Models_Results/svm/`.

Interpretación: MAE logarítmico es la métrica primaria de selección; RMSE asigna más peso a errores grandes. Las métricas en unidades describen el error tras aplicar `expm1`. El número de vectores de soporte describe el ajuste, pero no mide calidad por sí solo.

Las ventas tienen una distribución de cola pesada: conviven productos con pocas ventas y otros con muchísimas. Estos valores extremos son posibles y relevantes para el negocio; no se consideran errores de captura solo por ser extremos y no se eliminan como outliers. En test, la categoría `>500` presenta MAE logarítmico de aproximadamente 0.94–0.99 y MAE en unidades de aproximadamente 1,903–2,099, mientras las categorías de menor venta tienen MAE logarítmico de aproximadamente 0.31–0.39. Esto evidencia que SVR tiene dificultad para representar la cola alta con las características disponibles. No es un fallo de ejecución ni justifica borrar esos casos: refleja la heterogeneidad real de ventas y el límite predictivo del modelo para este problema.

Limitaciones: `bought_in_last_month` es el punto medio del rango informado por Amazon, no un conteo exacto. Los datos heredan las imputaciones y transformaciones previas del pipeline de limpieza. La búsqueda usa una muestra agrupada de train para reducir el costo computacional; los modelos finales se ajustan con todo el train. Las variantes v3/v4 contienen filas con precio imputado en entrenamiento, mientras el test común mantiene productos con precio observado.

In [7]:
metrics = {
    "model": "Support Vector Regression (SVR)",
    "target": TARGET,
    "selection_metric": "MAE en escala logarítmica",
    "random_state": SEED,
    "features_by_variant": {f"v{v}": datasets[v]["features"] for v in VARIANTS},
    "group_features": group_features,
    "best_params": best_params,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "n_support_vectors": support_vectors,
    "train_rows": {f"v{v}": int(len(datasets[v]["train_idx"])) for v in VARIANTS},
    "test_rows": {f"v{v}": int(len(datasets[v]["test_idx"])) for v in VARIANTS},
    "search": {
        "sample_size_requested": SEARCH_SAMPLE_SIZE,
        "cv": f"StratifiedGroupKFold(n_splits={CV_FOLDS}, shuffle=True)",
        "scoring": "neg_mean_absolute_error",
        "parameter_grid": PARAM_GRID,
        "seconds_by_variant": search_seconds,
    },
    "final_fit_seconds": fit_seconds,
    "limitations": [
        "El objetivo es el punto medio del rango de compras reportado por Amazon.",
        "La búsqueda usa una muestra agrupada de entrenamiento por el costo computacional de SVR.",
        "Las variantes v3/v4 incorporan filas con precio imputado en entrenamiento.",
        "Los datasets contienen imputaciones y transformaciones hechas antes de la partición.",
    ],
}
with (RESULTS_DIR / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print(f"Métricas: {RESULTS_DIR / 'metrics.json'}")
print(f"Búsqueda: {RESULTS_DIR / 'hyperparameter_search.csv'}")
print(f"Predicciones: {RESULTS_DIR / 'test_predictions.csv'}")
print("Modelos serializados: svr_v1_final.joblib a svr_v4_final.joblib")
print("Las métricas finales de test no se usaron para seleccionar hiperparámetros.")

Métricas: /home/barrera/Documentos/Udea/2026-2/Modelos y simulación/Trabajo de amazon/Amazon product sales/Modelos entrenados/Models_Results/svm/metrics.json
Búsqueda: /home/barrera/Documentos/Udea/2026-2/Modelos y simulación/Trabajo de amazon/Amazon product sales/Modelos entrenados/Models_Results/svm/hyperparameter_search.csv
Predicciones: /home/barrera/Documentos/Udea/2026-2/Modelos y simulación/Trabajo de amazon/Amazon product sales/Modelos entrenados/Models_Results/svm/test_predictions.csv
Modelos serializados: svr_v1_final.joblib a svr_v4_final.joblib
Las métricas finales de test no se usaron para seleccionar hiperparámetros.
